# ML-Based Short-Term Futures Price Prediction Using Order Book Data

This notebook implements a machine learning pipeline to predict short-term futures prices using real-time order book data fetched via WebSocket connections.

## 1. Import Required Libraries

Import libraries for data fetching, processing, and modeling.

In [1]:
import websocket
import json
import pandas as pd
import numpy as np
from datetime import datetime
import time
import threading
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Conv2D, Flatten
import matplotlib.pyplot as plt

2025-11-14 18:02:21.858800: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### GPU Availability Check

In [2]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Establish WebSocket Connection for Order Book Data

Set up a WebSocket connection to Binance Futures API to stream real-time order book data.

In [3]:
# WebSocket URL for Binance Futures Order Book
SYMBOL = 'btcusdt'
DEPTH_LEVELS = 20
UPDATE_SPEED = 100  # ms

ws_url = f'wss://fstream.binance.com/ws/{SYMBOL}@depth{DEPTH_LEVELS}@{UPDATE_SPEED}ms'

# Global variables to store data
order_book_data = []
data_lock = threading.Lock()

def on_message(ws, message):
    data = json.loads(message)
    with data_lock:
        order_book_data.append({
            'timestamp': datetime.now(),
            'bids': data['b'],
            'asks': data['a'],
            'lastUpdateId': data['u']
        })

def on_error(ws, error):
    print(f"Error: {error}")

def on_close(ws, close_status_code, close_msg):
    print("Connection closed")

def on_open(ws):
    print("Connection opened")

# Function to start WebSocket in a thread
def start_websocket():
    ws = websocket.WebSocketApp(ws_url,
                                on_message=on_message,
                                on_error=on_error,
                                on_close=on_close,
                                on_open=on_open)
    ws.run_forever()

# Start WebSocket in background thread
ws_thread = threading.Thread(target=start_websocket)
ws_thread.daemon = True
ws_thread.start()

Connection opened


## 3. Fetch and Parse Order Book Snapshots

Collect and parse incoming WebSocket messages into structured data.

### Fetch Historical Order Book Data (Optional for More Data)

Use CCXT to fetch order book snapshots over a period for backtesting.

In [4]:
import ccxt

# Fetch historical order book data using CCXT
exchange = ccxt.binance({
    'options': {'defaultType': 'future'},
})

historical_data = []
num_snapshots = 100  # Number of snapshots to fetch

for i in range(num_snapshots):
    try:
        order_book = exchange.fetch_order_book('BTC/USDT', limit=20)
        historical_data.append({
            'timestamp': datetime.now(),
            'bids': order_book['bids'],
            'asks': order_book['asks']
        })
        time.sleep(1)  # Wait 1 second between fetches to avoid rate limits
    except Exception as e:
        print(f"Error fetching data: {e}")
        break

# Convert to DataFrame
df_hist = pd.DataFrame(historical_data)
if not df_hist.empty:
    df_hist['bid_prices'], df_hist['bid_quantities'] = zip(*df_hist['bids'].apply(lambda x: ([p for p, q in x], [q for p, q in x])))
    df_hist['ask_prices'], df_hist['ask_quantities'] = zip(*df_hist['asks'].apply(lambda x: ([p for p, q in x], [q for p, q in x])))
    print(f"Fetched {len(df_hist)} historical snapshots")
else:
    print("No historical data fetched")

# Optionally, combine with real-time data
# df = pd.concat([df, df_hist], ignore_index=True)

Fetched 100 historical snapshots


In [5]:
# Collect data for a longer period to get more samples
time.sleep(60)  # Wait for 1 minute of data

with data_lock:
    df = pd.DataFrame(order_book_data)

# Parse bids and asks into separate columns
def parse_levels(levels):
    prices = [float(level[0]) for level in levels]
    quantities = [float(level[1]) for level in levels]
    return prices, quantities

df['bid_prices'], df['bid_quantities'] = zip(*df['bids'].apply(parse_levels))
df['ask_prices'], df['ask_quantities'] = zip(*df['asks'].apply(parse_levels))

print(f"Collected {len(df)} order book snapshots")
print(df.head())

Collected 1456 order book snapshots
                   timestamp  \
0 2025-11-14 18:02:24.238791   
1 2025-11-14 18:02:24.397545   
2 2025-11-14 18:02:24.509699   
3 2025-11-14 18:02:24.635777   
4 2025-11-14 18:02:24.741804   

                                                bids  \
0  [[96859.90, 5.587], [96859.80, 0.065], [96859....   
1  [[96859.90, 5.594], [96859.80, 0.065], [96859....   
2  [[96859.90, 6.573], [96859.80, 0.065], [96859....   
3  [[96866.90, 9.697], [96866.60, 0.470], [96866....   
4  [[96873.20, 2.862], [96872.90, 0.589], [96872....   

                                                asks   lastUpdateId  \
0  [[96860.00, 1.917], [96860.10, 0.004], [96860....  9191390072119   
1  [[96860.00, 1.917], [96860.10, 0.004], [96860....  9191390081561   
2  [[96860.00, 0.071], [96860.10, 0.004], [96860....  9191390106492   
3  [[96867.00, 0.004], [96867.30, 0.006], [96867....  9191390153406   
4  [[96873.80, 0.003], [96874.00, 0.051], [96874....  9191390194061   

       

## 4. Feature Engineering from Order Book Data

Compute features like bid-ask imbalance, mid-price, spread, and volatility metrics.

In [6]:
print(f"df shape: {df.shape}")
print(df.head() if not df.empty else "df is empty")

df shape: (1456, 8)
                   timestamp  \
0 2025-11-14 18:02:24.238791   
1 2025-11-14 18:02:24.397545   
2 2025-11-14 18:02:24.509699   
3 2025-11-14 18:02:24.635777   
4 2025-11-14 18:02:24.741804   

                                                bids  \
0  [[96859.90, 5.587], [96859.80, 0.065], [96859....   
1  [[96859.90, 5.594], [96859.80, 0.065], [96859....   
2  [[96859.90, 6.573], [96859.80, 0.065], [96859....   
3  [[96866.90, 9.697], [96866.60, 0.470], [96866....   
4  [[96873.20, 2.862], [96872.90, 0.589], [96872....   

                                                asks   lastUpdateId  \
0  [[96860.00, 1.917], [96860.10, 0.004], [96860....  9191390072119   
1  [[96860.00, 1.917], [96860.10, 0.004], [96860....  9191390081561   
2  [[96860.00, 0.071], [96860.10, 0.004], [96860....  9191390106492   
3  [[96867.00, 0.004], [96867.30, 0.006], [96867....  9191390153406   
4  [[96873.80, 0.003], [96874.00, 0.051], [96874....  9191390194061   

                       

In [21]:
df.head(10)

,timestamp,bids,asks,lastUpdateId,bid_prices,bid_quantities,ask_prices,ask_quantities,imbalance,mid_price,spread,mid_price_volatility,price_change,order_book_image
9,2025-11-14 18:02:25.235666,"[[96873.90, 1.861], [96873.80, 0.066], [96873....","[[96874.00, 3.487], [96874.10, 0.004], [96874....",9191390324469,"[96873.9, 96873.8, 96873.4, 96873.3, 96872.9, ...","[1.861, 0.066, 0.003, 0.002, 0.003, 0.035, 0.0...","[96874.0, 96874.1, 96874.2, 96874.4, 96874.5, ...","[3.487, 0.004, 0.002, 0.003, 0.005, 0.005, 0.0...",-0.280397,96873.95,0.1,6.605320,0.0,"[[1.861, 3.487], [0.066, 0.004], [0.003, 0.002..."
10,2025-11-14 18:02:25.334999,"[[96873.90, 1.914], [96873.80, 0.066], [96873....","[[96874.00, 1.954], [96874.10, 0.004], [96874....",9191390333165,"[96873.9, 96873.8, 96873.4, 96873.3, 96872.9, ...","[1.914, 0.066, 0.003, 0.002, 0.003, 0.083, 0.0...","[96874.0, 96874.1, 96874.2, 96874.5, 96875.0, ...","[1.954, 0.004, 0.002, 0.005, 0.005, 0.003, 0.0...",0.017953,96873.95,0.1,5.921076,0.0,"[[1.914, 1.954], [0.066, 0.004], [0.003, 0.002..."
11,2025-11-14 18:02:25.445843,"[[96873.90, 1.889], [96873.80, 0.066], [96873....","[[96874.00, 1.946], [96874.10, 0.004], [96874....",9191390343398,"[96873.9, 96873.8, 96873.4, 96873.3, 96872.9, ...","[1.889, 0.066, 0.003, 0.004, 0.003, 0.083, 0.0...","[96874.0, 96874.1, 96874.2, 96874.5, 96875.0, ...","[1.946, 0.004, 0.002, 0.005, 0.005, 0.003, 0.0...",0.013866,96873.95,0.1,4.704516,0.0,"[[1.889, 1.946], [0.066, 0.004], [0.003, 0.002..."
12,2025-11-14 18:02:25.585196,"[[96873.90, 1.905], [96873.80, 0.066], [96873....","[[96874.00, 1.642], [96874.10, 0.004], [96874....",9191390357272,"[96873.9, 96873.8, 96873.4, 96873.3, 96872.9, ...","[1.905, 0.066, 0.003, 0.004, 0.003, 0.083, 0.0...","[96874.0, 96874.1, 96874.2, 96874.5, 96875.0, ...","[1.642, 0.004, 0.002, 0.005, 0.005, 0.003, 0.0...",0.095538,96873.95,0.1,2.202328,0.0,"[[1.905, 1.642], [0.066, 0.004], [0.003, 0.002..."
13,2025-11-14 18:02:25.687604,"[[96873.90, 1.578], [96873.80, 0.066], [96873....","[[96874.00, 1.964], [96874.10, 0.004], [96874....",9191390368684,"[96873.9, 96873.8, 96873.4, 96873.3, 96872.9, ...","[1.578, 0.066, 0.003, 0.004, 0.003, 0.002, 0.0...","[96874.0, 96874.1, 96874.2, 96874.5, 96875.0, ...","[1.964, 0.004, 0.002, 0.005, 0.005, 0.003, 0.0...",-0.099174,96873.95,0.1,0.142302,0.0,"[[1.578, 1.964], [0.066, 0.004], [0.003, 0.002..."
14,2025-11-14 18:02:25.845340,"[[96873.90, 1.906], [96873.80, 0.066], [96873....","[[96874.00, 1.964], [96874.10, 0.004], [96874....",9191390381609,"[96873.9, 96873.8, 96873.4, 96873.3, 96872.9, ...","[1.906, 0.066, 0.003, 0.004, 0.003, 0.002, 0.0...","[96874.0, 96874.1, 96874.2, 96874.5, 96875.0, ...","[1.964, 0.004, 0.002, 0.005, 0.005, 0.003, 0.0...",-0.013333,96873.95,0.1,0.000000,0.0,"[[1.906, 1.964], [0.066, 0.004], [0.003, 0.002..."
15,2025-11-14 18:02:25.991681,"[[96873.90, 1.149], [96873.80, 0.066], [96873....","[[96874.00, 3.160], [96874.10, 0.004], [96874....",9191390399964,"[96873.9, 96873.8, 96873.4, 96873.3, 96872.9, ...","[1.149, 0.066, 0.003, 0.004, 0.003, 0.014, 0.0...","[96874.0, 96874.1, 96874.2, 96874.5, 96875.0, ...","[3.16, 0.004, 0.002, 0.005, 0.005, 0.06, 0.083...",-0.447481,96873.95,0.1,0.000000,0.0,"[[1.149, 3.16], [0.066, 0.004], [0.003, 0.002]..."
16,2025-11-14 18:02:26.090692,"[[96873.90, 1.031], [96873.80, 0.066], [96873....","[[96874.00, 3.347], [96874.10, 0.004], [96874....",9191390411967,"[96873.9, 96873.8, 96873.4, 96873.3, 96872.9, ...","[1.031, 0.066, 0.003, 0.004, 0.003, 0.002, 0.0...","[96874.0, 96874.1, 96874.2, 96874.5, 96875.0, ...","[3.347, 0.004, 0.002, 0.005, 0.005, 0.06, 0.08...",-0.505155,96873.95,0.1,0.000000,0.0,"[[1.031, 3.347], [0.066, 0.004], [0.003, 0.002..."
17,2025-11-14 18:02:26.197646,"[[96873.90, 1.342], [96873.80, 0.066], [96873....","[[96874.00, 3.158], [96874.10, 0.004], [96874....",9191390422950,"[96873.9, 96873.8, 96873.4, 96873.3, 96872.9, ...","[1.342, 0.066, 0.003, 0.004, 0.003, 0.002, 0.0...","[96874.0, 96874.1, 96874.2, 96874.5

In [7]:
# Feature Engineering
def calculate_imbalance(bid_quantities, ask_quantities, k=10):
    bid_sum = sum(bid_quantities[:k])
    ask_sum = sum(ask_quantities[:k])
    return (bid_sum - ask_sum) / (bid_sum + ask_sum) if (bid_sum + ask_sum) > 0 else 0

def calculate_mid_price(bid_prices, ask_prices):
    return (bid_prices[0] + ask_prices[0]) / 2

def calculate_spread(bid_prices, ask_prices):
    return ask_prices[0] - bid_prices[0]

df['imbalance'] = df.apply(lambda row: calculate_imbalance(row['bid_quantities'], row['ask_quantities']), axis=1)
df['mid_price'] = df.apply(lambda row: calculate_mid_price(row['bid_prices'], row['ask_prices']), axis=1)
df['spread'] = df.apply(lambda row: calculate_spread(row['bid_prices'], row['ask_prices']), axis=1)

# Rolling volatility of mid_price
df['mid_price_volatility'] = df['mid_price'].rolling(window=10).std()

# Price change (target for next period)
df['price_change'] = df['mid_price'].shift(-1) - df['mid_price']

print(df[['timestamp', 'imbalance', 'mid_price', 'spread', 'mid_price_volatility', 'price_change']].head())

                   timestamp  imbalance  mid_price  spread  \
0 2025-11-14 18:02:24.238791   0.490135   96859.95     0.1   
1 2025-11-14 18:02:24.397545   0.489045   96859.95     0.1   
2 2025-11-14 18:02:24.509699   0.969316   96859.95     0.1   
3 2025-11-14 18:02:24.635777   0.996336   96866.95     0.1   
4 2025-11-14 18:02:24.741804   0.989013   96873.50     0.6   

   mid_price_volatility  price_change  
0                   NaN          0.00  
1                   NaN          0.00  
2                   NaN          7.00  
3                   NaN          6.55  
4                   NaN          0.45  


## 5. Prepare Dataset for Modeling

Clean the data, create binary labels for price movements, and split into training/validation sets.

In [8]:
# Prepare Dataset
df = df.dropna()  # Drop rows with NaN

# Features
features = ['imbalance', 'spread', 'mid_price_volatility']
X = df[features]

# Target: Binary classification (1 if price increases, 0 otherwise)
y = (df['price_change'] > 0).astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}, Test set size: {len(X_test)}")

Training set size: 1156, Test set size: 290


In [9]:
# Prepare Dataset
df = df.dropna()  # Drop rows with NaN

# Features
features = ['imbalance', 'spread', 'mid_price_volatility']
X = df[features]

# Target: Binary classification (1 if price increases, 0 otherwise)
y = (df['price_change'] > 0).astype(int)

# Check class distribution
print("Overall target distribution:")
print(y.value_counts())
print(f"Percentage of 1s: {y.mean():.2%}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}, Test set size: {len(X_test)}")
print(f"Train target distribution: {y_train.value_counts()}")
print(f"Test target distribution: {y_test.value_counts()}")

Overall target distribution:
price_change
0    1348
1      98
Name: count, dtype: int64
Percentage of 1s: 6.78%
Training set size: 1156, Test set size: 290
Train target distribution: price_change
0    1077
1      79
Name: count, dtype: int64
Test target distribution: price_change
0    271
1     19
Name: count, dtype: int64


## 6. Train Supervised Learning Model

Train XGBoost or Random Forest on the engineered features.

In [10]:
# Train XGBoost
model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

Model Accuracy: 0.92


In [11]:
# Train XGBoost with class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100, scale_pos_weight=class_weight_dict[1]/class_weight_dict[0])
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Alternative: Use sample weights
# sample_weights = np.where(y_train == 1, class_weight_dict[1], class_weight_dict[0])
# model.fit(X_train, y_train, sample_weight=sample_weights)

Model Accuracy: 0.89


## 7. Implement Deep Learning Model

Build and train an LSTM model to capture temporal patterns.

### CNN Model for Order Book as Image

Treat the order book as a 2D matrix (bids and asks) and use CNN to extract spatial features.

In [12]:
# Create 2D representation of order book
def create_order_book_image(bid_quantities, ask_quantities, max_levels=20):
    # Pad or truncate to max_levels
    bids = np.array(bid_quantities[:max_levels])
    asks = np.array(ask_quantities[:max_levels])
    if len(bids) < max_levels:
        bids = np.pad(bids, (0, max_levels - len(bids)), 'constant')
    if len(asks) < max_levels:
        asks = np.pad(asks, (0, max_levels - len(asks)), 'constant')
    # Stack as 2 channels: bids and asks
    image = np.stack([bids, asks], axis=-1)  # Shape: (max_levels, 2)
    return image

# Apply to dataframe
df['order_book_image'] = df.apply(lambda row: create_order_book_image(row['bid_quantities'], row['ask_quantities']), axis=1)

# Prepare data for CNN
X_images = np.stack(df['order_book_image'].values)
X_images = X_images[..., np.newaxis]  # Add channel dimension: (n, 20, 2, 1)
y_images = y.values[:len(X_images)]  # Align with images

X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(X_images, y_images, test_size=0.2, random_state=42)

# Build CNN model
model_cnn = Sequential()
model_cnn.add(Conv2D(32, (3, 1), activation='relu', input_shape=(20, 2, 1)))
model_cnn.add(Flatten())
model_cnn.add(Dense(64, activation='relu'))
model_cnn.add(Dense(1, activation='sigmoid'))

model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_cnn.fit(X_train_img, y_train_img, epochs=10, batch_size=32, validation_data=(X_test_img, y_test_img))

/home/misango/anaconda3/envs/research_env/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1763114720.545570  126536 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9800 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


Epoch 1/10


2025-11-14 18:05:22.065071: I external/local_xla/xla/service/service.cc:163] XLA service 0x797404007050 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-14 18:05:22.065110: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2025-11-14 18:05:22.088419: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-14 18:05:22.260255: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91501
2025-11-14 18:05:22.279923: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-14 18:05:22.279923: I e

35/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9321 - loss: 0.4814 

I0000 00:00:1763114724.410058  130520 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.9308 - loss: 0.3677 - val_accuracy: 0.9345 - val_loss: 0.2491
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.9308 - loss: 0.3677 - val_accuracy: 0.9345 - val_loss: 0.2491
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9334 - loss: 0.2133 - val_accuracy: 0.9448 - val_loss: 0.2184
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9334 - loss: 0.2133 - val_accuracy: 0.9448 - val_loss: 0.2184
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9377 - loss: 0.1906 - val_accuracy: 0.9414 - val_loss: 0.2122
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9377 - loss: 0.1906 - val_accuracy: 0.9414 - val_loss: 0.2122
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9386 - loss: 0.1836 - val_accuracy: 0.9448 - val_loss: 0.2078
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9386 - loss: 0.1836 - val_accuracy: 0.9448 - val_loss: 0.2078
E

In [13]:
# Create 2D representation of order book
def create_order_book_image(bid_quantities, ask_quantities, max_levels=20):
    # Pad or truncate to max_levels
    bids = np.array(bid_quantities[:max_levels])
    asks = np.array(ask_quantities[:max_levels])
    if len(bids) < max_levels:
        bids = np.pad(bids, (0, max_levels - len(bids)), 'constant')
    if len(asks) < max_levels:
        asks = np.pad(asks, (0, max_levels - len(asks)), 'constant')
    # Stack as 2 channels: bids and asks
    image = np.stack([bids, asks], axis=-1)  # Shape: (max_levels, 2)
    return image

# Apply to dataframe
df['order_book_image'] = df.apply(lambda row: create_order_book_image(row['bid_quantities'], row['ask_quantities']), axis=1)

# Prepare data for CNN
X_images = np.stack(df['order_book_image'].values)
X_images = X_images[..., np.newaxis]  # Add channel dimension: (n, 20, 2, 1)
y_images = y.values[:len(X_images)]  # Align with images

X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(X_images, y_images, test_size=0.2, random_state=42)

# Class weights for CNN
class_weights_cnn = compute_class_weight('balanced', classes=np.unique(y_train_img), y=y_train_img)
class_weight_dict_cnn = {0: class_weights_cnn[0], 1: class_weights_cnn[1]}

# Build CNN model
model_cnn = Sequential()
model_cnn.add(Conv2D(32, (3, 1), activation='relu', input_shape=(20, 2, 1)))
model_cnn.add(Flatten())
model_cnn.add(Dense(64, activation='relu'))
model_cnn.add(Dense(1, activation='sigmoid'))

model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_cnn.fit(X_train_img, y_train_img, epochs=10, batch_size=32, validation_data=(X_test_img, y_test_img), class_weight=class_weight_dict_cnn)

Epoch 1/10


/home/misango/anaconda3/envs/research_env/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.6003 - loss: 0.6059 - val_accuracy: 0.7483 - val_loss: 0.5336
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.6003 - loss: 0.6059 - val_accuracy: 0.7483 - val_loss: 0.5336
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7699 - loss: 0.4979 - val_accuracy: 0.7414 - val_loss: 0.5433
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7699 - loss: 0.4979 - val_accuracy: 0.7414 - val_loss: 0.5433
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7569 - loss: 0.4657 - val_accuracy: 0.7759 - val_loss: 0.4120
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7569 - loss: 0.4657 - val_accuracy: 0.7759 - val_loss: 0.4120
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8201 - loss: 0.4447 - val_accuracy: 0.7345 - val_loss: 0.5576
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8201 - loss: 0.4447 - val_accuracy: 0.7345 - val_loss: 0.5576
E

In [14]:
# Prepare data for LSTM (sequences)
sequence_length = 10
X_seq = []
y_seq = []
for i in range(len(X) - sequence_length):
    X_seq.append(X.iloc[i:i+sequence_length].values)
    y_seq.append(y.iloc[i+sequence_length])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

# Build LSTM model
model_lstm = Sequential()
model_lstm.add(LSTM(50, input_shape=(sequence_length, len(features))))
model_lstm.add(Dense(1, activation='sigmoid'))

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32, validation_data=(X_test_seq, y_test_seq))

Epoch 1/10


/home/misango/anaconda3/envs/research_env/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8571 - loss: 0.4772 - val_accuracy: 0.9271 - val_loss: 0.3035
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8571 - loss: 0.4772 - val_accuracy: 0.9271 - val_loss: 0.3035
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9329 - loss: 0.2638 - val_accuracy: 0.9271 - val_loss: 0.2514
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9329 - loss: 0.2638 - val_accuracy: 0.9271 - val_loss: 0.2514
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9329 - loss: 0.2353 - val_accuracy: 0.9271 - val_loss: 0.2528
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9329 - loss: 0.2353 - val_accuracy: 0.9271 - val_loss: 0.2528
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9329 - loss: 0.2292 - val_accuracy: 0.9271 - val_loss: 0.2586
Epoch 5/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9329 - loss: 0.2292 - val_accuracy: 0.9271 - val_loss: 0.2586
E

In [15]:
# Prepare data for LSTM (sequences)
sequence_length = 10
X_seq = []
y_seq = []
for i in range(len(X) - sequence_length):
    X_seq.append(X.iloc[i:i+sequence_length].values)
    y_seq.append(y.iloc[i+sequence_length])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

# Class weights for LSTM
class_weights_lstm = compute_class_weight('balanced', classes=np.unique(y_train_seq), y=y_train_seq)
class_weight_dict_lstm = {0: class_weights_lstm[0], 1: class_weights_lstm[1]}

# Build LSTM model
model_lstm = Sequential()
model_lstm.add(LSTM(50, input_shape=(sequence_length, len(features))))
model_lstm.add(Dense(1, activation='sigmoid'))

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32, validation_data=(X_test_seq, y_test_seq), class_weight=class_weight_dict_lstm)

Epoch 1/10


/home/misango/anaconda3/envs/research_env/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5052 - loss: 0.6808 - val_accuracy: 0.4306 - val_loss: 0.7355
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5052 - loss: 0.6808 - val_accuracy: 0.4306 - val_loss: 0.7355
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4930 - loss: 0.6374 - val_accuracy: 0.4826 - val_loss: 0.7145
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4930 - loss: 0.6374 - val_accuracy: 0.4826 - val_loss: 0.7145
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5200 - loss: 0.6105 - val_accuracy: 0.5382 - val_loss: 0.6391
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5200 - loss: 0.6105 - val_accuracy: 0.5382 - val_loss: 0.6391
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5697 - loss: 0.5993 - val_accuracy: 0.5139 - val_loss: 0.6780
Epoch 5/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5697 - loss: 0.5993 - val_accuracy: 0.5139 - val_loss: 0.6780
E

## 8. Generate Price Predictions

Use the trained models to predict on new data.

In [16]:
# Predictions with XGBoost
predictions_xgb = model.predict(X_test)

# Predictions with LSTM
predictions_lstm = (model_lstm.predict(X_test_seq) > 0.5).astype(int).flatten()

# Predictions with CNN
predictions_cnn = (model_cnn.predict(X_test_img) > 0.5).astype(int).flatten()

print("XGBoost Predictions:", predictions_xgb[:10])
print("LSTM Predictions:", predictions_lstm[:10])
print("CNN Predictions:", predictions_cnn[:10])

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
XGBoost Predictions: [0 0 0 0 0 0 0 0 0 0]
LSTM Predictions: [1 0 0 0 1 0 0 1 0 1]
CNN Predictions: [1 0 0 0 1 0 1 0 0 1]
XGBoost Predictions: [0 0 0 0 0 0 0 0 0 0]
LSTM Predictions: [1 0 0 0 1 0 0 1 0 1]
CNN Predictions: [1 0 0 0 1 0 1 0 0 1]


In [17]:
# Predictions with XGBoost
predictions_xgb = model.predict(X_test)

# Predictions with LSTM
predictions_lstm = (model_lstm.predict(X_test_seq) > 0.5).astype(int).flatten()

# Predictions with CNN
predictions_cnn = (model_cnn.predict(X_test_img) > 0.5).astype(int).flatten()

print("XGBoost Predictions:", predictions_xgb[:10])
print("LSTM Predictions:", predictions_lstm[:10])
print("CNN Predictions:", predictions_cnn[:10])

# Diagnostic: Check class distribution
print("\nTarget distribution in test set:")
print(y_test.value_counts())

# Diagnostic: Accuracy on test set
from sklearn.metrics import classification_report
print("\nXGBoost Classification Report:")
print(classification_report(y_test, predictions_xgb))

# For LSTM and CNN, since they use different splits, check separately if needed
# But for simplicity, focus on XGBoost

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
XGBoost Predictions: [0 0 0 0 0 0 0 0 0 0]
LSTM Predictions: [1 0 0 0 1 0 0 1 0 1]
CNN Predictions: [1 0 0 0 1 0 1 0 0 1]

Target distribution in test set:
price_change
0    271
1     19
Name: count, dtype: int64

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       271
           1       0.25      0.32      0.28        19

    accuracy                           0.89       290
   macro avg       0.60      0.62      0.61       290
weighted avg       0.91      0.89      0.90       290

XGBoost Predictions: [0 0 0 0 0 0 0 0 0 0]
LSTM Predictions: [1 0 0 0 1 0 0 1 0 1]
CNN Predictions: [1 0 0 0 1 0 1 0 0 1]

Target distribution in test set:
price_change
0    271
1     19
Name: count, dtype: int64

XGBoost Classification Report:
              precision    

In [18]:
# Predictions with XGBoost
predictions_xgb = model.predict(X_test)

# Predictions with LSTM
predictions_lstm = (model_lstm.predict(X_test_seq) > 0.5).astype(int).flatten()

# Predictions with CNN
predictions_cnn = (model_cnn.predict(X_test_img) > 0.5).astype(int).flatten()

print("XGBoost Predictions:", predictions_xgb[:10])
print("LSTM Predictions:", predictions_lstm[:10])
print("CNN Predictions:", predictions_cnn[:10])

# Diagnostic: Check class distribution
print("\nTarget distribution in test set:")
print(y_test.value_counts())

# Diagnostic: Accuracy on test set
from sklearn.metrics import classification_report
print("\nXGBoost Classification Report:")
print(classification_report(y_test, predictions_xgb))

# Check prediction distribution
print("\nPrediction distribution for XGBoost:")
print(pd.Series(predictions_xgb).value_counts())

# For LSTM and CNN, since they use different splits, check separately if needed
# But for simplicity, focus on XGBoost

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
XGBoost Predictions: [0 0 0 0 0 0 0 0 0 0]
LSTM Predictions: [1 0 0 0 1 0 0 1 0 1]
CNN Predictions: [1 0 0 0 1 0 1 0 0 1]

Target distribution in test set:
price_change
0    271
1     19
Name: count, dtype: int64

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       271
           1       0.25      0.32      0.28        19

    accuracy                           0.89       290
   macro avg       0.60      0.62      0.61       290
weighted avg       0.91      0.89      0.90       290


Prediction distribution for XGBoost:
0    266
1     24
Name: count, dtype: int64
XGBoost Predictions: [0 0 0 0 0 0 0 0 0 0]
LSTM Predictions: [1 0 0 0 1 0 0 1 0 1]
CNN Predictions: [1 0 0 0 1 0 1 0 0 1]

Target distribution in test set:
price_change
0    271
1     19
Nam

## 9. Simulate Trading Execution

Implement a simple simulation of trade execution based on predictions and calculate potential profits.

In [19]:
# Simple Trading Simulation
# Assume we trade based on XGBoost predictions
capital = 10000
position = 0  # 1 for long, -1 for short
entry_price = 0

profits = []

for i in range(len(predictions_xgb)):
    pred = predictions_xgb[i]
    actual_change = df.iloc[len(X_train) + i]['price_change']
    current_price = df.iloc[len(X_train) + i]['mid_price']
    
    if position == 0:
        if pred == 1:  # Predict up, go long
            position = 1
            entry_price = current_price
        elif pred == 0:  # Predict down, go short
            position = -1
            entry_price = current_price
    else:
        # Close position
        if position == 1:
            profit = (current_price - entry_price) * 100  # Assume 1 lot
        else:
            profit = (entry_price - current_price) * 100
        profits.append(profit)
        capital += profit
        position = 0

total_profit = sum(profits)
print(f"Total Simulated Profit: {total_profit:.2f}, Final Capital: {capital:.2f}")

Total Simulated Profit: -3275.00, Final Capital: 6725.00


In [20]:
# Simple Trading Simulation
# Assume we trade based on XGBoost predictions
capital = 10000
position = 0  # 1 for long, -1 for short
entry_price = 0

profits = []
trades = []

for i in range(len(predictions_xgb)):
    pred = predictions_xgb[i]
    actual_change = df.iloc[len(X_train) + i]['price_change']
    current_price = df.iloc[len(X_train) + i]['mid_price']
    
    if position == 0:
        if pred == 1:  # Predict up, go long
            position = 1
            entry_price = current_price
            trades.append(('long', entry_price))
        elif pred == 0:  # Predict down, go short
            position = -1
            entry_price = current_price
            trades.append(('short', entry_price))
    else:
        # Close position on next signal or end
        if position == 1:
            profit = (current_price - entry_price) * 100  # Assume 1 lot
        else:
            profit = (entry_price - current_price) * 100
        profits.append(profit)
        capital += profit
        position = 0  # Close position

# If still in position at end, close it
if position != 0:
    last_price = df.iloc[len(X_train) + len(predictions_xgb) - 1]['mid_price']
    if position == 1:
        profit = (last_price - entry_price) * 100
    else:
        profit = (entry_price - last_price) * 100
    profits.append(profit)
    capital += profit

total_profit = sum(profits)
print(f"Total Simulated Profit: {total_profit:.2f}, Final Capital: {capital:.2f}")
print(f"Number of trades: {len(trades)}")
print(f"Average profit per trade: {total_profit / len(trades) if trades else 0:.2f}")

# Compare to buy-and-hold
initial_price = df.iloc[len(X_train)]['mid_price']
final_price = df.iloc[len(X_train) + len(predictions_xgb) - 1]['mid_price']
buy_hold_profit = (final_price - initial_price) * 100
print(f"Buy-and-hold profit: {buy_hold_profit:.2f}")
print(f"Strategy vs Buy-and-hold: {'Better' if total_profit > buy_hold_profit else 'Worse'}")

Total Simulated Profit: -3275.00, Final Capital: 6725.00
Number of trades: 145
Average profit per trade: -22.59
Buy-and-hold profit: 8870.00
Strategy vs Buy-and-hold: Worse
